# Nguyễn Thành Trí - MSSV: 6451071079
## Bài 2 (Trang 15-17) - Câu 6: Tiền xử lý các thuộc tính thiếu dữ liệu và xóa bỏ các thuộc tính không cần thiết
1. Trích xuất thông tin boong tàu (`deck`) từ thuộc tính `cabin`, sau đó xóa bỏ `cabin`.
2. Xóa bỏ các thuộc tính không cần thiết: `ticket`, `boat`, `body`, `home.dest`.
3. Điền giá trị thiếu cho thuộc tính tuổi (`age`) bằng các số ngẫu nhiên trong khoảng `[mean - std, mean + std]`.
4. Điền giá trị thiếu cho thuộc tính cổng lên tàu (`embarked`) bằng giá trị xuất hiện nhiều nhất (`'S'`).
5. Điền giá trị thiếu cho thuộc tính giá vé (`fare`) nếu còn thiếu.

In [ ]:
# Nguyễn Thành Trí - 6451071079
# Bài 2 (15-17) - Câu 6: Tiền xử lý thiếu dữ liệu và loại bỏ thuộc tính không cần thiết

import os
import re
import numpy as np
import pandas as pd

print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

# Đọc dữ liệu
def get_data_path(rel_path):
    candidates = [
        rel_path,
        os.path.join('..', rel_path),
        os.path.join('..', '..', rel_path),
        os.path.join('datasets', os.path.basename(rel_path)),
        os.path.join('..', 'datasets', os.path.basename(rel_path)),
        os.path.join('..', '..', 'datasets', os.path.basename(rel_path))
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'Không tìm thấy file dữ liệu: {rel_path}')

csv_path = get_data_path('titanic.csv')
df = pd.read_csv(csv_path, na_values=['?'])
df.columns = df.columns.str.lower()
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df['fare'] = pd.to_numeric(df['fare'], errors='coerce')

# 1. Trích xuất boong tàu (deck) từ cabin trước khi xóa cabin
deck_map = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7, "U": 8}
df['cabin'] = df['cabin'].fillna("U0")
df['deck'] = df['cabin'].map(lambda x: re.compile(r"([a-zA-Z]+)").search(x).group())
df['deck'] = df['deck'].map(deck_map).fillna(0).astype(int)
df = df.drop(['cabin'], axis=1)

# 2. Xóa các thuộc tính không cần thiết / gây rò rỉ nhãn
cols_to_drop = ['ticket', 'boat', 'body', 'home.dest']
df = df.drop([c for c in cols_to_drop if c in df.columns], axis=1)

# 3. Điền giá trị thiếu cho thuộc tính tuổi (age) bằng số ngẫu nhiên trong [mean - std, mean + std]
mean = df["age"].mean()
std = df["age"].std()
is_null = df["age"].isnull().sum()

np.random.seed(42)  # Cố định seed ngẫu nhiên
rand_age = np.random.randint(mean - std, mean + std, size=is_null)

age_slice = df["age"].copy()
age_slice[np.isnan(age_slice)] = rand_age
df["age"] = age_slice
df["age"] = df["age"].astype(int)

# 4. Điền giá trị thiếu cho cổng lên tàu (embarked) bằng mode 'S'
common_value = 'S'
df['embarked'] = df['embarked'].fillna(common_value)

# 5. Điền giá trị thiếu cho giá vé (fare)
df['fare'] = df['fare'].fillna(0)

print("\n--- KIỂM TRA LẠI SỐ LƯỢNG GIÁ TRỊ RỖNG SAU BƯỚC TIỀN XỬ LÝ ---")
print(df.isnull().sum())

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079

--- KIỂM TRA LẠI SỐ LƯỢNG GIÁ TRỊ RỖNG SAU BƯỚC TIỀN XỬ LÝ ---
pclass      0
survived    0
name        0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
deck        0
dtype: int64
